In [1]:
from IPython import get_ipython
from IPython.display import display

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.6 MB/s eta 0:00:00


In [4]:
import os
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.svm import SVR
from skopt import gp_minimize
from skopt.space import Real, Categorical, Integer
from scipy.stats import zscore
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from scipy.stats import zscore

In [29]:
# Load dataset
features_train = pd.read_csv("/content/drive/MyDrive/Kuliah/Proposal/New/dengue_features_train.csv")
labels_train = pd.read_csv("/content/drive/MyDrive/Kuliah/Proposal/New/dengue_labels_train.csv")
features_test = pd.read_csv("/content/drive/MyDrive/Kuliah/Proposal/New/dengue_features_test.csv")

In [30]:
# Store 'week_start_date' separately for each city
week_start_date_train_sj = features_train.loc[features_train["city"] == "sj", 'week_start_date'].copy()
week_start_date_train_iq = features_train.loc[features_train["city"] == "iq", 'week_start_date'].copy()
week_start_date_test = features_test['week_start_date'].copy()

In [31]:
# Drop 'week_start_date' column from features dataframes
features_train.drop(columns=["week_start_date"], inplace=True)
features_test.drop(columns=["week_start_date"], inplace=True)

In [32]:
# Merge features and labels
train_data = features_train.merge(labels_train, on=["city", "year", "weekofyear"])

In [33]:
# Split dataset by city
sj_train = train_data[train_data["city"] == "sj"].drop(columns=["city"])
iq_train = train_data[train_data["city"] == "iq"].drop(columns=["city"])
features_test_sj = features_test[features_test["city"] == "sj"].drop(columns=['city'])
features_test_iq = features_test[features_test["city"] == "iq"].drop(columns=['city'])

Preprocessing

In [34]:
## Data Preprocessing

def preprocess_data(df_train, df_test, city_name):
    """Handles missing values, outlier detection (based on train), and standardization."""

    # Handle missing values (fill with median from training data)
    median_values = df_train.median(numeric_only=True)
    df_train.fillna(median_values, inplace=True)
    df_test.fillna(median_values, inplace=True) # Use training median for test data

    # Outlier Detection and Removal (based on training data)
    # Calculate Z-scores for training data (excluding 'total_cases' if it's the target)
    features_for_outlier_detection = df_train.drop(columns=['total_cases'] if 'total_cases' in df_train.columns else [], errors='ignore')
    z_scores_train = np.abs(zscore(features_for_outlier_detection))
    outliers_train_mask = (z_scores_train > 3).any(axis=1)
    outlier_rows_train = df_train.index[outliers_train_mask].tolist()
    print(f"Jumlah data dengan outlier di {city_name} train: {len(outlier_rows_train)}")

    # **Remove outliers from the training data**
    df_train_cleaned = df_train.drop(outlier_rows_train)
    print(f"Jumlah data train setelah outlier removal di {city_name}: {len(df_train_cleaned)}")


    # Separate features and target from the CLEANED training data
    if 'total_cases' in df_train_cleaned.columns:
        X_train = df_train_cleaned.drop(columns=['total_cases', 'year'], errors='ignore') # Drop year as it might not be a good feature for SVR/PCA
        y_train = df_train_cleaned['total_cases']
    else:
        X_train = df_train_cleaned.drop(columns=['year'], errors='ignore')
        y_train = None # Should not happen for train data

    X_test = df_test.drop(columns=['year'], errors='ignore') # Drop year from test data

    # Apply StandardScaler (fit on CLEANED train, transform CLEANED train and test)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test) # Use the same scaler fitted on train

    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

    # Apply PCA (fit on CLEANED train, transform CLEANED train and test)
    pca = PCA(n_components=12) # Choose number of components
    X_train_pca = pca.fit_transform(X_train_scaled_df)
    X_test_pca = pca.transform(X_test_scaled_df) # Use the same PCA fitted on train

    X_train_pca_df = pd.DataFrame(X_train_pca, index=X_train.index)
    X_test_pca_df = pd.DataFrame(X_test_pca, index=X_test.index)

    return X_train_pca_df, y_train, X_test_pca_df

In [35]:
# Preprocess data for San Juan and Iquitos
X_train_sj_pca, y_train_sj, X_test_sj_pca = preprocess_data(sj_train, features_test_sj.copy(), "San Juan")
X_train_iq_pca, y_train_iq, X_test_iq_pca = preprocess_data(iq_train, features_test_iq.copy(), "Iquitos")


print("\nProcessed data shapes:")
print("San Juan Train Features (PCA):", X_train_sj_pca.shape)
print("San Juan Train Target:", y_train_sj.shape)
print("San Juan Test Features (PCA):", X_test_sj_pca.shape)
print("Iquitos Train Features (PCA):", X_train_iq_pca.shape)
print("Iquitos Train Target:", y_train_iq.shape)
print("Iquitos Test Features (PCA):", X_test_iq_pca.shape)

Jumlah data dengan outlier di San Juan train: 87
Jumlah data train setelah outlier removal di San Juan: 849
Jumlah data dengan outlier di Iquitos train: 51
Jumlah data train setelah outlier removal di Iquitos: 469

Processed data shapes:
San Juan Train Features (PCA): (849, 12)
San Juan Train Target: (849,)
San Juan Test Features (PCA): (260, 12)
Iquitos Train Features (PCA): (469, 12)
Iquitos Train Target: (469,)
Iquitos Test Features (PCA): (156, 12)


Model Training and Evaluation

In [36]:
from sklearn.linear_model import LinearRegression

def train_and_evaluate_linear_regression(X_train, y_train, X_test, y_test, city_name):
    """Trains and evaluates a basic Linear Regression model."""
    lr_model = LinearRegression()

    start_time = time.time()
    lr_model.fit(X_train, y_train)
    end_time = time.time()
    execution_time = end_time - start_time

    y_pred = lr_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n--- Basic Linear Regression Results ({city_name}) ---")
    print(f"Mean Absolute Error: {mae}")
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Training Time: {execution_time:.4f} seconds")

    return lr_model # Return the trained model

In [38]:
# Split training data for validation (for evaluation metrics)
X_train_sj_split, X_test_sj_split, y_train_sj_split, y_test_sj_split = train_test_split(
    X_train_sj_pca, y_train_sj, test_size=0.2, random_state=42)
X_train_iq_split, X_test_iq_split, y_train_iq_split, y_test_iq_split = train_test_split(
    X_train_iq_pca, y_train_iq, test_size=0.2, random_state=42)

In [39]:
# Train and evaluate basic Linear Regression
basic_lr_sj = train_and_evaluate_linear_regression(X_train_sj_split, y_train_sj_split, X_test_sj_split, y_test_sj_split, "San Juan (Basic)")
basic_lr_iq = train_and_evaluate_linear_regression(X_train_iq_split, y_train_iq_split, X_test_iq_split, y_test_iq_split, "Iquitos (Basic)")


--- Basic Linear Regression Results (San Juan (Basic)) ---
Mean Absolute Error: 26.890554693648685
Root Mean Squared Error: 45.38127703388772
Training Time: 0.0129 seconds

--- Basic Linear Regression Results (Iquitos (Basic)) ---
Mean Absolute Error: 7.843458042780081
Root Mean Squared Error: 14.663485645130795
Training Time: 0.0060 seconds
